# Chapter 1 — Build and analyze a coupled grounded LC

Engineer course · source candidate · CONVERGING

# Chapter 1 — Build and analyze a coupled grounded LC

This complete Chapter is the clean-kernel execution unit. Its five web
Lessons are shorter reading views over these same ordered source
fragments; they are not independent notebooks. QMD is the editable
authority, while `chapter.ipynb` is a generated zero-output transport
artifact.

## Lesson 1 — Build the coupled resonator

### Start from a clean engineer workspace

Clone the repository, install the locked development environment, and
start Jupyter from the repository root:

``` bash
git clone --branch develop --single-branch https://github.com/OrPenStrike/scnsim.git
cd scnsim
uv sync --locked
uv run --with jupyterlab jupyter lab
```

`jupyterlab` is ephemeral development tooling in that command; it is not
added to the project or its lock. If your editor already supplies a
Jupyter frontend, open the Notebook there after `uv sync --locked`
instead.

Open `examples/engineer/chapter-01/chapter.ipynb`, select the project’s
`.venv` Python kernel, restart it, and use **Run All**. The Chapter is
complete in one clean kernel; the individual web Lessons are reading
views and deliberately share Chapter state. On a cold machine, the first
numerical request may prepare the locked Julia runtime and take longer
than later requests. Its workspace is evidence owned by this Run, not a
cache to copy into another project.

### Define the one value we will change

This first circuit is a lumped grounded parallel LC resonator,
capacitively coupled to one terminated measurement Port. Capacitance is
the sole independent input because Lesson 5 will select another physical
value on this same Plan. The inductor, coupler, and Port impedance
remain fixed literals.

We use a Subsystem because the capacitor, inductor, shared signal bus,
and local ground form one owned LC unit that is easier to review and
later reuse. The coupler and measurement Port stay parent-owned because
they connect that unit to its environment. A Subsystem is an explicit
ownership choice, not a wrapper required around every component.

In [ ]:
import numpy as np
from IPython.display import display

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    CircuitRun,
    DiagonalRootSpec,
    DirectSolveSpec,
    ParameterDefinitions,
    ParameterSet,
    ParameterSpec,
    ReductionPipeline,
    Theme,
    components,
    units as u,
)

inputs = ParameterDefinitions(id="engineer_lc_design")
capacitance = inputs.parameter(
    id="capacitance",
    baseline=110.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)

Allocating `capacitance` does not add it to a circuit. The physical
binding is the capacitor factory field in the next cell. `110 fF` is the
baseline, not a mutable global value.

### Author the grounded LC inside a child boundary

In [ ]:
plan = CircuitPlan(id="engineer_coupled_lc")
resonator = plan.subsystem(id="resonator")

capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=capacitance)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
resonator_bus = resonator.bus(id="signal")
resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)
resonator_terminal = resonator.expose_pin(
    id="terminal",
    at=resonator_bus,
)
resonator_coordinate = resonator.expose_coordinate(
    id="signal",
    at=resonator_bus,
)

The exposed Pin is the parent wiring boundary. The exposed Coordinate is
the analysis handle for the same child signal bus. Neither declaration
adds an electrical node, component, or wire, and the parent never
reaches into the child’s private bus.

### Couple the child directly to one terminated Port

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
coupling_capacitor = plan.add(
    components.capacitor(id="coupling_capacitor", capacitance=6.0 * u.fF)
)
plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_capacitor,),
    end=resonator_terminal,
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

The 6 fF element ends directly at the public child Pin. There is no
analysis-only root bus or zero-component Link on the resonator side. The
terminated Port owns its declared 50 ohm reference load to the canonical
ground.

## Lesson 2 — Review the authored circuit

**Prerequisite:** run all Lesson 1 cells in the aggregate Chapter
Notebook.

The authoring diagram is a projection of the Plan just declared. It does
not select a numerical View or execute a solver.

In [ ]:
diagram = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
    )
)

In [ ]:
diagram.show()

The independent audit checks whether visible connectivity and
containment agree with the captured Plan while retaining invisible
source declarations as source evidence.

In [ ]:
diagram.audit.show()

## Lesson 3 — Read the one-port S11 response

**Prerequisite:** run Lessons 1–2 in the aggregate Chapter Notebook.

`CircuitRun` seals this Plan for analysis and owns its evidence
workspace. The frequency grid is an explicit 401-point request from 5.75
GHz through 6.25 GHz.

In [ ]:
frequencies = np.linspace(5.75, 6.25, 401) * u.GHz
run = CircuitRun(
    plan=plan,
    workspace="workspaces/engineer-chapter-01",
)
direct_spec = DirectSolveSpec(frequencies=frequencies)

In [ ]:
baseline_response = run.solve(
    run.original,
    direct_spec,
    parameters=ParameterSet(),
)

The public presentation shows both magnitude and phase from the returned
typed Result. For this ideal lossless one-port model, `|S11|` is
approximately 0 dB across the grid; the phase still carries the readable
response feature. A magnitude dip is therefore not a resonance
definition for this example. The fixed ±0.05 dB display range keeps
floating-point roundoff from being visually autoscaled into a false
feature; it does not round or change the response data. The phase is
shown as returned, including its ordinary wrapped-phase discontinuity;
that jump is angle presentation, not a numerical failure. The horizontal
values are Hz, and Matplotlib displays the `1e9` scale factor for this
GHz-range request.

In [ ]:
baseline_s11_figure = baseline_response.s.show(magnitude="db")
baseline_s11_figure.axes[0].set_ylim(-0.05, 0.05)
baseline_s11_figure

## Lesson 4 — Ask for the loaded root

**Prerequisite:** run Lessons 1–3 in the aggregate Chapter Notebook.

The exposed child Coordinate selects the same authored resonator signal
bus for analysis. Retaining it derives a View; it does not alter the
Plan or diagram.

In [ ]:
root_view = run.original.reduce(
    ReductionPipeline().retain(resonator_coordinate)
)
root_spec = DiagonalRootSpec(
    coordinate=resonator_coordinate,
    root_hint=6.0 * u.GHz,
)

In [ ]:
baseline_root = run.evaluate(
    root_view,
    root_spec,
    parameters=ParameterSet(),
)

In [ ]:
display(baseline_root.frequency)
display(baseline_root.linewidth)
baseline_root.show()

The 6 GHz hint chooses the deterministic root basin; it is not the
answer or a search interval. The returned frequency and linewidth
describe the loaded selected operator diagonal. They are neither an
S-parameter dip nor the unloaded formula `1 / (2π√(LC))`.

## Lesson 5 — Change capacitance on the same Plan

**Prerequisite:** run Lessons 1–4 in the aggregate Chapter Notebook.

A `ParameterSet` selects 120 fF for the already-bound capacitor. It does
not mutate the Plan’s 110 fF baseline or construct a second circuit.

In [ ]:
selected_parameters = ParameterSet({capacitance: 120.0 * u.fF})

Use the same frequency grid, sealed Run, original View, and Direct
request.

In [ ]:
selected_response = run.solve(
    run.original,
    direct_spec,
    parameters=selected_parameters,
)

In [ ]:
selected_s11_figure = selected_response.s.show(magnitude="db")
selected_s11_figure.axes[0].set_ylim(-0.05, 0.05)
selected_s11_figure

Evaluate the same loaded-root request at the selected point.

In [ ]:
selected_root = run.evaluate(
    root_view,
    root_spec,
    parameters=selected_parameters,
)

In [ ]:
display(selected_root.frequency)
display(selected_root.linewidth)
selected_root.show()

The generated quantity table reports both loaded Results beside the
separate unloaded LC calculation. It does not infer a dip from the
nearly flat ideal one-port magnitude.

Changing the selected physical capacitance shifts the phase feature and
loaded root without redefining the circuit. The Plan, request, parameter
point, and each Result retain separate identities.